# Нэмэлт 8 багцын талбарын нэрийг НЭГТГЭХ

**ArcGIS Notebook дотор ажиллуулна** (AGOL → Notebook → Python 3). Токен
автоматаар ирнэ — гараар оруулах шаардлагагүй.

---

## Асуудал

Шинэ 8 багц (`6.1 · 6.2 · 6.4 · 5.1 · 5.2 · 5.3 · 5.4 · 10`) нь хуучин 10
багцаас **талбарын нэрээрээ** зөрж байна. Нийтлэг 17 талбар нь ижил нэртэй
боловч сүүлийн 5 нь **ГУРВАН өөр бичлэгтэй**:

| Утга | 6.1·6.2·6.4 | 5.3 | 5.1·5.2·10 | **Зорилтот** |
|---|---|---|---|---|
| Бөглөсөн огноо | `Бөглөсөн_огноо` | `Бөглөгдсөн_огноо` | `buglusun_ognoo` | `buglusun_ognoo` |
| Дэс дугаар | `Дэс_дугаар` | `Дэс_дугаар` | `des_dugaar` | `Des_dugaar` |
| Хамаарал | `Хамаарал` | `Хамаарал` | `hamaaral` | `Hamaaral` |
| Инж. обьём | `Инженерийн_төлөвлөсөн_объём` | (ижил) | `inj_tuluvlusun_obyom` | `Инженерийн_төлөвлөсөн_обьём` |
| Шинэчлэгдсэн | `Шинэчлэгдсэн_огноо` | `Шинэчлэгдсэн_огноо` | `shinechlegdsen_ognoo` | `Шинэчлэгдсэн_огноо` |

Зорилтот нэр нь **хуучин 10 багцынхтай ЯГ ижил** — тэгвэл код нэг замаар
уншина.

---

## ⚠️ ЮУ ХИЙХГҮЙ ВЭ — БЛОКИЙН БАГАНА

Хуучин 10 багц нь **барилга** тул блокоор задардаг (`F5_1_гүйцэтгэл`,
`F5_1_obyem`, `F5_1_geree_ehleh` … 12 блок × 6 багана).

Шинэ 8 нь **өөр төрлийн ажил** тул блокгүй — `Ажил_гүйцэтгэл` гэсэн НЭГ
баганатай. Энэ нь **алдаа биш, зөв бүтэц**.

Тиймээс энэ notebook хоосон `F5_*` багана **нэмэхгүй**. Нэмбэл:
- 12 × 6 = 72 хоосон багана үүсч, хэзээ ч дүүрэхгүй
- «Хуваарь» модуль тэднийг блок гэж үзэж, хоосон мөр зурна
- Гүйцэтгэлийн хувь `null ÷ null` болж утгагүй болно

Блокгүй багцыг уншихыг **кодоор** шийднэ (`bagts.pkg.ts` — доод тайлбарыг үз).

---

## Аргачлал — ЯАГААД нэр СОЛИХГҮЙ, ШИНЭ талбар НЭМЭХ вэ

ArcGIS-д талбарын нэрийг **шууд өөрчлөх API байхгүй**. Тиймээс:

1. Зорилтот нэртэй **шинэ** талбар нэмнэ (`addToDefinition`)
2. Хуучин талбараас утгыг нь **хуулна** (`calculate`)
3. Хуучныг нь **үлдээнэ** — устгавал буцаах арга байхгүй

Хуучин талбар үлдэх нь хор хөнөөлгүй: код зорилтот нэрийг уншина, хуучин нь
зүгээр л ашиглагдахгүй хэвтэнэ. Хожим итгэлтэй болсон үедээ гараар устгаж
болно.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  1. ТОХИРГОО
# ══════════════════════════════════════════════════════════════════════
import json
import urllib.parse
import urllib.request

from arcgis.gis import GIS

# ⚠️ Notebook дотор токен АВТОМАТААР ирнэ — гараар бичихгүй
gis = GIS("home")
TOKEN = gis._con.token
print("нэвтэрсэн:", gis.properties.user.username)

ORG = "https://services.arcgis.com/HJzgwvlNIXssnQar/arcgis/rest/services"

# Шинэ 8 багц
SHEETS = [
    "Bagts_6_1", "Bagts_6_2", "Bagts_6_4",
    "Bagts_5_1", "Bagts_5_2", "Bagts_5_3", "Bagts_5_4",
    "Bagts_10",
]

# ⚠️ ЭХЛЭЭД ҮРГЭЛЖ True-ЭЭР АЖИЛЛУУЛ — юу болохыг хараад дараа нь False
DRY = True

print(f"багц: {len(SHEETS)} · DRY={DRY}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  2. ЗОРИЛТОТ ТАЛБАРУУД ба ТЭДГЭЭРИЙН ЭХ СУРВАЛЖ
# ══════════════════════════════════════════════════════════════════════
#
# `src` нь ХУВИЛБАРУУДЫН жагсаалт — аль нь байвал тэрнээс хуулна.
# ⚠️ Дараалал нь ЧУХАЛ: эхэнд байгаа нь эрхэмлэгдэнэ.
#
# ⚠️ Зорилтот нэр нь ХУУЧИН 10 БАГЦЫНХТАЙ ЯГ ИЖИЛ байх ЁСТОЙ — тэр нь
#    `src/modules/sheet/bagts.pkg.ts`-д хатуу бичигдсэн (`gun`, `Des_dugaar`,
#    `Hamaaral`, `buglusun_ognoo`). Зөрвөл код олохгүй.

TARGETS = [
    {
        "name": "buglusun_ognoo",
        "type": "esriFieldTypeDate",
        "alias": "Бөглөсөн огноо",
        "src": ["Бөглөсөн_огноо", "Бөглөгдсөн_огноо"],
        "why": "АРХИВЫН түлхүүр — нийтлэх бүрд агшин ялгана",
    },
    {
        "name": "Des_dugaar",
        "type": "esriFieldTypeInteger",
        "alias": "Дэс дугаар",
        "src": ["Дэс_дугаар", "des_dugaar"],
        "why": "агшин дамжсан ТОГТВОРТОЙ түлхүүр (№ нь давтагддаг)",
    },
    {
        "name": "Hamaaral",
        "type": "esriFieldTypeString",
        "alias": "Хамаарал",
        "length": 255,
        "src": ["Хамаарал", "hamaaral"],
        "why": "уялдаа холбоос — «18FS3,22SS-5»",
    },
    {
        "name": "Инженерийн_төлөвлөсөн_обьём",
        "type": "esriFieldTypeDouble",
        "alias": "Инженерийн төлөвлөсөн обьём",
        # ⚠️ «объём» (ХАТУУГИЙН тэмдэг) vs «обьём» (ЗӨӨЛНИЙ) — өөр тэмдэгт!
        "src": ["Инженерийн_төлөвлөсөн_объём", "inj_tuluvlusun_obyom"],
        "why": "батлагдсаны дараа бичигддэг зорилт (obyemBatlah.ts)",
    },
    {
        "name": "gun",
        "type": "esriFieldTypeSmallInteger",
        "alias": "Шатлал",
        "src": [],  # ⚠️ эх сурвалжгүй — ХООСОН нэмнэ, нийтлэхэд дүүрнэ
        "why": "мөрийн модны гүн 0-4; FillNew.publish дүүргэнэ",
    },
]

for t in TARGETS:
    src = " ← ".join(t["src"]) if t["src"] else "(хоосон)"
    print(f"  {t['name']:<32} {src}")
    print(f"  {'':<32} {t['why']}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  3. ТУСЛАХ ФУНКЦУУД
# ══════════════════════════════════════════════════════════════════════

def post(url, params):
    """ArcGIS REST руу POST.

    ⚠️ АЛДАА нь HTTP 200-аар ирдэг — биеийг ЗААВАЛ шалгана. `res.ok` хангалтгүй.
    """
    params = {**params, "f": "json", "token": TOKEN}
    data = urllib.parse.urlencode(params).encode()
    with urllib.request.urlopen(urllib.request.Request(url, data=data)) as r:
        out = json.loads(r.read().decode())
    if isinstance(out, dict) and "error" in out:
        raise RuntimeError(out["error"].get("message", out["error"]))
    return out


def get(url, params=None):
    params = {**(params or {}), "f": "json", "token": TOKEN}
    full = f"{url}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(full) as r:
        out = json.loads(r.read().decode())
    if isinstance(out, dict) and "error" in out:
        raise RuntimeError(out["error"].get("message", out["error"]))
    return out


def layer_urls(sheet):
    """(унших, админ) хос замыг буцаана.

    ⚠️ ТАЛБАРЫН БҮТЭЦ өөрчлөх нь `/rest/admin/services/…` руу ханддаг —
       энгийн `/rest/services/…` дээр `499 Token Required` буцаана.
    """
    read = f"{ORG}/{sheet}/FeatureServer/0"
    admin = read.replace("/rest/services/", "/rest/admin/services/")
    return read, admin


print("бэлэн")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  4. ОДООГИЙН БАЙДЛЫГ ШАЛГАХ  (юу ч ӨӨРЧЛӨХГҮЙ)
# ══════════════════════════════════════════════════════════════════════

state = {}
print(f"{'БАГЦ':<14}{'мөр':>7}  талбарууд")
print("=" * 78)

for sheet in SHEETS:
    read, _ = layer_urls(sheet)
    try:
        meta = get(read)
        cnt = get(f"{read}/query", {"where": "1=1", "returnCountOnly": "true"})
    except Exception as e:
        print(f"{sheet:<14} ⛔ {e}")
        state[sheet] = None
        continue

    names = [f["name"] for f in meta.get("fields", [])]
    state[sheet] = names

    marks = []
    for t in TARGETS:
        if t["name"] in names:
            marks.append(f"✅{t['name']}")
        else:
            found = next((s for s in t["src"] if s in names), None)
            marks.append(f"➕{t['name']}←{found}" if found else f"➕{t['name']}(хоосон)")

    print(f"{sheet:<14}{cnt.get('count', '?'):>7}  " + "  ".join(marks))

print("\n✅ = аль хэдийн бий  ·  ➕ = нэмэгдэнэ")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  5. ТАЛБАР НЭМЭХ  +  УТГА ХУУЛАХ
# ══════════════════════════════════════════════════════════════════════
#
# ⚠️ ДАРААЛАЛ ЧУХАЛ: эхлээд БҮХ талбарыг нэмээд, дараа нь утгыг хуулна.
#    Нэг талбар нэмэх бүрд `calculate` дуудвал давхарга удаа дараа түгжигдэж,
#    том хуудсанд удаан болно.

added, copied, skipped, failed = 0, 0, 0, 0

for sheet in SHEETS:
    names = state.get(sheet)
    if names is None:
        print(f"{sheet:<14} ⛔ алгассан (уншигдаагүй)")
        continue

    read, admin = layer_urls(sheet)

    # ── 5a. Дутуу талбаруудыг НЭГ дуудлагаар нэмнэ ──
    need = [t for t in TARGETS if t["name"] not in names]
    if not need:
        print(f"{sheet:<14} ✅ бүх талбар бэлэн")
        skipped += 1
        continue

    new_fields = []
    for t in need:
        # ⚠️ `nullable` ЗААВАЛ: нэмэгдэх агшинд бүх хуучин мөр хоосон байна
        f = {
            "name": t["name"],
            "type": t["type"],
            "alias": t["alias"],
            "nullable": True,
            "editable": True,
        }
        if "length" in t:
            f["length"] = t["length"]
        new_fields.append(f)

    if DRY:
        lst = ", ".join(t["name"] for t in need)
        print(f"{sheet:<14} ➕ нэмэгдэнэ: {lst}")
        added += len(need)
        continue

    try:
        post(f"{admin}/0/addToDefinition",
             {"addToDefinition": json.dumps({"fields": new_fields})})
        print(f"{sheet:<14} ✅ {len(need)} талбар нэмэгдэв")
        added += len(need)
    except Exception as e:
        print(f"{sheet:<14} ⛔ нэмэхэд алдаа: {e}")
        failed += 1
        continue

    # ── 5b. Хуучин талбараас утга хуулна ──
    for t in need:
        src = next((s for s in t["src"] if s in names), None)
        if not src:
            continue  # эх сурвалжгүй (`gun`) — хоосон үлдэнэ
        try:
            # ⚠️ `calculate` нь ЦЭГЦТЭЙ SQL биш, ИЛЭРХИЙЛЭЛ авдаг.
            #    Талбарын нэр кирилл тул хашилтанд хийнэ.
            post(f"{read}/calculate", {
                "where": "1=1",
                "calcExpression": json.dumps(
                    [{"field": t["name"], "sqlExpression": f'"{src}"'}]),
            })
            print(f"{'':<14}    ↳ {t['name']} ← {src}")
            copied += 1
        except Exception as e:
            print(f"{'':<14}    ⛔ {t['name']} ← {src}: {e}")
            failed += 1

tag = "(DRY — юу ч хийсэнгүй) " if DRY else ""
print(f"\n{tag}нэмсэн {added} · хуулсан {copied} · бэлэн {skipped} · алдаа {failed}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  6. БАТАЛГААЖУУЛАЛТ — утга ҮНЭХЭЭР хуулагдсан уу?
# ══════════════════════════════════════════════════════════════════════
#
# ⚠️ `calculate` нь алдаагүй буцаад ч НЭГ Ч МӨР хөндөөгүй байж болно
#    (жишээ нь эх талбар бүхэлдээ `null`). Тиймээс ТООЛЖ шалгана.

print(f"{'БАГЦ':<14}{'нийт':>6}", end="")
for t in TARGETS:
    print(f"{t['name'][:12]:>14}", end="")
print("\n" + "=" * 86)

for sheet in SHEETS:
    if state.get(sheet) is None:
        continue
    read, _ = layer_urls(sheet)
    try:
        total = get(f"{read}/query",
                    {"where": "1=1", "returnCountOnly": "true"}).get("count", 0)
    except Exception as e:
        print(f"{sheet:<14} ⛔ {e}")
        continue

    print(f"{sheet:<14}{total:>6}", end="")
    for t in TARGETS:
        try:
            n = get(f"{read}/query", {
                "where": f'"{t["name"]}" IS NOT NULL',
                "returnCountOnly": "true",
            }).get("count", 0)
            print(f"{n:>14}", end="")
        except Exception:
            print(f"{'⛔':>14}", end="")
    print()

print("\n⚠️ `gun` нь 0 байх нь ХЭВИЙН — «Гүйцэтгэл бөглөх» дээр багц бүрийг")
print("   НЭГ УДАА нийтлэхэд `FillNew.publish` мөр бүрд шатлалыг нь бичнэ.")

---

## Notebook-ийн ДАРАА — код талд хийх зүйл

### 1. Хуваалцах тохиргоо (5.4)

`Bagts_5_4` нь **токенгүй уншигдахгүй** байсан (`Token Required`). Бусад 7
нээлттэй. AGOL дээр тэр item-ийн Share-ийг бусадтай ижил болгоно — эс
бөгөөс портал дээр тэр багц нээгдэхгүй.

### 2. `bagts.pkg.ts`-д 8 мөр нэмэх

```ts
{ key: "b61", group: 'Багц 6.1', floors: 9,
  label: tr('Багц 6.1'), url: `${HJ}/Bagts_6_1/FeatureServer/0` },
// … 6.2 · 6.4 · 5.1 · 5.2 · 5.3 · 5.4 · 10
```

### 3. ⚠️ БЛОКГҮЙ БАГЦЫГ УНШИХ — кодын засвар ЗААВАЛ

Энэ notebook блокийн багана **нэмэхгүй** (дээрх шалтгаанаар). Тиймээс шинэ 8
багцад `Schema`-гийн дараах талбарууд **хоосон массив** болно:

```
bld · act · plan · obyem · start · end · gStart · gEnd  →  []
```

Одоогийн код нь блок БАЙХ гэж үздэг тул шалгах шаардлагатай газрууд:

| Файл | Юу болох вэ |
|---|---|
| `bagts.pkg.ts` | `bld.length === 0` үед бүдүүвч зөв буцаах уу |
| `Huvaari.tsx` | блокоор чирдэг — блокгүй үед юу харуулах вэ |
| `bagtsSheet.computeAll` | `E = C × J` блок бүрээр — нэг баганад яах вэ |
| `FillNew.tsx` | блокийн багана зурдаг |

Шинэ 8 багцад гүйцэтгэл нь `Ажил_гүйцэтгэл` гэсэн НЭГ баганад байх тул
«блок 1 ширхэг» гэж үзэх нь хамгийн бага өөрчлөлттэй зам байж магадгүй —
гэхдээ **энэ шийдвэрийг код уншиж баталсны дараа** гаргана.

### 4. Шалгуур

`npm test` дотор `bagts.check.mjs` нь «7/7 багц нийтлэгдсэн» гэж тоолдог —
шинэ багц нэмэгдэхэд тэр тоо өөрчлөгдөнө.